In [7]:
import numpy as np
import pandas as pd
import seaborn as sns
import datetime as dt
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedShuffleSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, balanced_accuracy_score, confusion_matrix, precision_recall_curve, roc_auc_score, accuracy_score, f1_score, r2_score, mean_absolute_error, mutual_info_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import LinearSVC, SVC
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

---
---

## 甚麼都不做, 直接跑隨機森林, 結論是還是要做features selection, 效果會較佳

In [11]:
secom = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\secom.csv', sep='\t')
np.random.seed(42)
X = secom.drop(columns=['Time', 'Pass/Fail'])
y = secom['Pass/Fail'].replace({-1:0, 1:1}).astype(int)

na_counts = X.isna().sum()
X = X.loc[:, na_counts<=550].copy()
uniq_counts = X.nunique(dropna=False)
X = X.loc[:, uniq_counts>=50].copy()
X = X.loc[:, ~X.T.duplicated(keep='first')].copy()

def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])  # [[TN, FP],[FN, TP]]
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    acc = accuracy_score(y_true, y_pred)
    sensitivity = TP/(TP+FN) if (TP+FN)>0 else np.nan
    specificity = TN/(TN+FP) if (TN+FP)>0 else np.nan
    FAR = FP/(FP+TN) if (FP+TN)>0 else np.nan
    GM = np.sqrt(sensitivity*specificity) if (not np.isnan(sensitivity) and not np.isnan(specificity)) else np.nan
    return {'cm':cm, 'acc':acc, 'FAR':FAR, 'sensitivity':sensitivity, 'specificity':specificity, 'GM':GM}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


all_metrics = []
for fold, (tr_idx, te_idx) in enumerate(skf.split(X_train, y_train), start=1):
    X_tr, X_te = X_train.iloc[tr_idx], X_train.iloc[te_idx]
    y_tr, y_te = y_train.iloc[tr_idx], y_train.iloc[te_idx]

    knn = KNNImputer(n_neighbors=10, weights='distance')
    X_tr_imputed = pd.DataFrame(knn.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    X_te_imputed = pd.DataFrame(knn.transform(X_te), columns=X_te.columns, index=X_te.index)

    smote = SMOTE(sampling_strategy=1, k_neighbors=5, random_state=42)
    X_balanced, y_balanced = smote.fit_resample(X_tr_imputed, y_tr)

    rf = RandomForestClassifier(n_estimators=1000, min_samples_leaf=15, max_depth=6, min_samples_split=20, random_state=42, n_jobs=-1)
    rf.fit(X_balanced, y_balanced)
    y_pred = rf.predict(X_te_imputed)

    m = compute_metrics(y_te.values, y_pred)
    all_metrics.append(m)

    print(f"fold={fold}, acc={m['acc']:.4f}, FAR={m['FAR']:.4f}")
    print("sensitivy = ", round(m["sensitivity"]*100, 2), "%")
    print("specificity = ", round(m["specificity"]*100, 2), "%")
    print("GM = ", round(m["GM"]*100, 2), "%")
    print("-"*40)

avg = {k: np.nanmean([m[k] for m in all_metrics]) for k in ["acc","FAR","sensitivity","specificity","GM"]}
print("[Mean over 10 folds]")
print(f"acc={avg['acc']:.4f}, FAR={avg['FAR']:.4f}")
print("sensitivy = ", round(avg["sensitivity"]*100, 2), "%")
print("specificity = ", round(avg["specificity"]*100, 2), "%")
print("GM = ", round(avg["GM"]*100, 2), "%")

fold=1, acc=0.8651, FAR=0.0684
sensitivy =  0.0 %
specificity =  93.16 %
GM =  0.0 %
----------------------------------------
fold=2, acc=0.9206, FAR=0.0342
sensitivy =  33.33 %
specificity =  96.58 %
GM =  56.74 %
----------------------------------------
fold=3, acc=0.8889, FAR=0.0513
sensitivy =  11.11 %
specificity =  94.87 %
GM =  32.47 %
----------------------------------------
fold=4, acc=0.9280, FAR=0.0342
sensitivy =  37.5 %
specificity =  96.58 %
GM =  60.18 %
----------------------------------------
fold=5, acc=0.9280, FAR=0.0171
sensitivy =  12.5 %
specificity =  98.29 %
GM =  35.05 %
----------------------------------------
fold=6, acc=0.9280, FAR=0.0427
sensitivy =  50.0 %
specificity =  95.73 %
GM =  69.18 %
----------------------------------------
fold=7, acc=0.8800, FAR=0.0598
sensitivy =  0.0 %
specificity =  94.02 %
GM =  0.0 %
----------------------------------------
fold=8, acc=0.9280, FAR=0.0256
sensitivy =  25.0 %
specificity =  97.44 %
GM =  49.35 %
-------------

---
---
## 二階段的logistic regression, 還是未獲得可用解（實務上沒收斂）
#### 可能要做降維或做L1, L2懲罰


In [46]:
secom = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\secom.csv', sep='\t')
np.random.seed(42)
X = secom.drop(columns=['Time', 'Pass/Fail'])
y = secom['Pass/Fail'].replace({-1:0, 1:1}).astype(int)

na_counts = X.isna().sum()
X = X.loc[:, na_counts<=550].copy()
uniq_counts = X.nunique(dropna=False)
X = X.loc[:, uniq_counts>=50].copy()
X = X.loc[:, ~X.T.duplicated(keep='first')].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

target_fold = 6
for fold, (tr_idx, te_idx) in enumerate(skf.split(X_train, y_train), start=1):
    if fold != target_fold:
        continue

    X_tr, X_te = X_train.iloc[tr_idx], X_train.iloc[te_idx]
    y_tr, y_te = y_train.iloc[tr_idx], y_train.iloc[te_idx]

    # 1) KNN 只 fit 在當折 train
    knn = KNNImputer(n_neighbors=10, weights='distance')
    X_tr_imp = pd.DataFrame(knn.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    X_te_imp = pd.DataFrame(knn.transform(X_te), columns=X_te.columns, index=X_te.index)

    l1 = LogisticRegression(penalty='l1', solver='liblinear', C=1.0, max_iter=2000, random_state=42)
    l1.fit(X_tr_imp, y_tr)
    coef = pd.Series(l1.coef_.ravel(), index=X_tr_imp.columns)
    selected = coef[coef != 0].index.tolist()
    if len(selected) == 0:
        selected = list(X_tr_imp.columns)

    # 3) 標準化（只用 train 估參數）
    scaler = StandardScaler()
    X_tr_sel = X_tr_imp[selected]
    X_te_sel = X_te_imp[selected]
    X_tr_z = pd.DataFrame(scaler.fit_transform(X_tr_sel), columns=selected, index=X_tr_sel.index)
    X_te_z = pd.DataFrame(scaler.transform(X_te_sel), columns=selected, index=X_te_sel.index)

    # 4) SMOTE 只在 train 上；採 0.7 打破對稱
    smote = SMOTE(sampling_strategy=0.7, k_neighbors=5, random_state=42)
    X_bal, y_bal = smote.fit_resample(X_tr_z, y_tr)

    # 6) 當折 train 上估計 Logit 並印 summary（無懲罰）
    Xb = sm.add_constant(X_bal, has_constant='add')
    res = sm.Logit(y_bal.values, Xb).fit(method='bfgs', maxiter=2000, disp=True)
    print("\n========== statsmodels Logit 第六折 summary ==========")
    print(res.summary())
    break

Optimization terminated successfully.
         Current function value: 0.000014
         Iterations: 845
         Function evaluations: 852
         Gradient evaluations: 852

========== statsmodels Logit 第六折 summary ==========
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                 1790
Model:                          Logit   Df Residuals:                     1606
Method:                           MLE   Df Model:                          183
Date:                Tue, 23 Sep 2025   Pseudo R-squ.:                   1.000
Time:                        22:14:02   Log-Likelihood:              -0.024192
converged:                       True   LL-Null:                       -1212.7
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------

---
---


## 目前最好的一次fold, 來做analysis, 也是一種方法

In [18]:
secom = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\secom.csv', sep='\t')
np.random.seed(42)
X = secom.drop(columns=['Time', 'Pass/Fail'])
y = secom['Pass/Fail'].replace({-1:0, 1:1}).astype(int)

na_counts = X.isna().sum()
X = X.loc[:, na_counts<=550].copy()
uniq_counts = X.nunique(dropna=False)
X = X.loc[:, uniq_counts>=50].copy()
X = X.loc[:, ~X.T.duplicated(keep='first')].copy()

def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])  # [[TN, FP],[FN, TP]]
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    acc = accuracy_score(y_true, y_pred)
    sensitivity = TP/(TP+FN) if (TP+FN)>0 else np.nan
    specificity = TN/(TN+FP) if (TN+FP)>0 else np.nan
    FAR = FP/(FP+TN) if (FP+TN)>0 else np.nan
    GM = np.sqrt(sensitivity*specificity) if (not np.isnan(sensitivity) and not np.isnan(specificity)) else np.nan
    return {'cm':cm, 'acc':acc, 'FAR':FAR, 'sensitivity':sensitivity, 'specificity':specificity, 'GM':GM}

def tri_class(y_proba_fail, low, high):
    """
    輸入 y_proba
    回傳 tri_class {-1:pass, 0:not sure, 1:Fail}
    """
    if not (0<=low<high<=1):
        raise ValueError('low必須<high, 且都在[0, 1]')
    tri = np.where(y_proba_fail<=low, -1, np.where(y_proba_fail>=high,1 ,0))
    return tri

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

selected_by_fold = {}
all_metrics = []
for fold, (tr_idx, te_idx) in enumerate(skf.split(X_train, y_train), start=1):
    X_tr, X_te = X_train.iloc[tr_idx], X_train.iloc[te_idx]
    y_tr, y_te = y_train.iloc[tr_idx], y_train.iloc[te_idx]

    knn = KNNImputer(n_neighbors=10, weights='distance')
    X_tr_imputed = pd.DataFrame(knn.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    X_te_imputed = pd.DataFrame(knn.transform(X_te), columns=X_te.columns, index=X_te.index)

    l1 = LogisticRegression(penalty='l1', solver='liblinear', C=1.0, max_iter=2000, random_state=42)
    l1.fit(X_tr_imputed, y_tr)

    coef = pd.Series(l1.coef_.ravel(), index=X_tr_imputed.columns)
    selected = coef[coef != 0].index.tolist()
    if len(selected) == 0:
        selected = list(X_tr_imputed.columns)

    selected_by_fold[fold] = selected

    coef_sel = coef.loc[selected]
    df_rank = (pd.DataFrame({"feature": coef_sel.index, "coef": coef_sel.values}).assign(l1_abs=lambda d: d["coef"].abs()).sort_values("l1_abs", ascending=False).reset_index(drop=True))
    df_rank["rank_l1_abs"] = np.arange(1, len(df_rank)+1)

    if fold == 6:
        print(f"[Fold 6] selected features = {len(selected)}")
        print("[Fold 6] order by |coef| (descending):")
        print(df_rank[["rank_l1_abs","feature","coef","l1_abs"]].head(30).to_string(index=False))
        print("（排序標準：|L1 係數| 由大到小）")

    X_tr_selected = X_tr_imputed[selected].copy()
    X_te_selected = X_te_imputed[selected].copy()

    smote = SMOTE(sampling_strategy=1, k_neighbors=5, random_state=42)
    X_balanced, y_balanced = smote.fit_resample(X_tr_selected, y_tr)

    rf = RandomForestClassifier(n_estimators=1000, min_samples_leaf=15, max_depth=6, min_samples_split=20, random_state=42, n_jobs=-1)
    rf.fit(X_balanced, y_balanced)
    y_pred = rf.predict(X_te_selected)

    if fold == 6:
        y_prob = rf.predict_proba(X_te_selected)[:, 1]
        tri = tri_class(y_prob, low=0.3, high=0.5)
        out = pd.DataFrame({"y_true": y_te.values, "y_pred_bin": y_pred, "p_fail": y_prob, "tri_class": tri}, index=X_te_selected.index)
        print(out.head())
        print("tri_class counts:\n", out["tri_class"].value_counts().sort_index())
        
        y_true_s = y_te.rename("y_true")
        y_pred_s = pd.Series(y_pred, index=X_te_selected.index, name="y_pred")
        fold6_secom_result = pd.concat([X_te_selected, y_true_s, y_pred_s], axis=1)
        print(fold6_secom_result.head(1))
        fold6_secom_test = secom.loc[X_te_selected.index].copy()
        fold6_secom_test["y_true"] = y_true_s
        fold6_secom_test["y_pred"] = y_pred_s
        print(confusion_matrix(y_te, y_pred, labels=[0, 1]))
        
    m = compute_metrics(y_te.values, y_pred)
    all_metrics.append(m)

    print(f"fold={fold}, acc={m['acc']:.4f}, FAR={m['FAR']:.4f}")
    print("sensitivy = ", round(m["sensitivity"]*100, 2), "%")
    print("specificity = ", round(m["specificity"]*100, 2), "%")
    print("GM = ", round(m["GM"]*100, 2), "%")
    print("-"*40)

avg = {k: np.nanmean([m[k] for m in all_metrics]) for k in ["acc","FAR","sensitivity","specificity","GM"]}
print("[Mean over 10 folds]")
print(f"acc={avg['acc']:.4f}, FAR={avg['FAR']:.4f}")
print("sensitivy = ", round(avg["sensitivity"]*100, 2), "%")
print("specificity = ", round(avg["specificity"]*100, 2), "%")
print("GM = ", round(avg["GM"]*100, 2), "%")

fold=1, acc=0.8651, FAR=0.0684
sensitivy =  0.0 %
specificity =  93.16 %
GM =  0.0 %
----------------------------------------
fold=2, acc=0.9048, FAR=0.0513
sensitivy =  33.33 %
specificity =  94.87 %
GM =  56.24 %
----------------------------------------
fold=3, acc=0.8730, FAR=0.0684
sensitivy =  11.11 %
specificity =  93.16 %
GM =  32.17 %
----------------------------------------
fold=4, acc=0.9040, FAR=0.0598
sensitivy =  37.5 %
specificity =  94.02 %
GM =  59.38 %
----------------------------------------
fold=5, acc=0.9200, FAR=0.0342
sensitivy =  25.0 %
specificity =  96.58 %
GM =  49.14 %
----------------------------------------
[Fold 6] selected features = 183
[Fold 6] order by |coef| (descending):
 rank_l1_abs feature      coef   l1_abs
           1    x446  2.168428 2.168428
           2    x178 -1.616071 1.616071
           3    x334  1.588997 1.588997
           4    x407  0.857853 0.857853
           5    x337 -0.711831 0.711831
           6     x15 -0.653757 0.653757
    

---
---

## 調整X, y在cv上的比例（進行中）

In [21]:
secom = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\DataSet\secom.csv', sep='\t')
np.random.seed(42)
X = secom.drop(columns=['Time', 'Pass/Fail'])
y = secom['Pass/Fail'].replace({-1:0, 1:1}).astype(int)

na_counts = X.isna().sum()
X = X.loc[:, na_counts<=550].copy()
uniq_counts = X.nunique(dropna=False)
X = X.loc[:, uniq_counts>=50].copy()
X = X.loc[:, ~X.T.duplicated(keep='first')].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# --- 關鍵：在訓練集上用 X 結構做分層鍵 ---
def build_strat_key_for_Xy(Xtr, ytr, n_splits=5, random_state=42):
    # 僅為「分折」用的管線：Impute→Scale→PCA(2)
    imputer = KNNImputer(n_neighbors=10, weights='distance')
    scaler  = StandardScaler()
    pca     = PCA(n_components=2, random_state=random_state)

    Xtr_ = Xtr.reset_index(drop=True)
    ytr_ = ytr.reset_index(drop=True)

    Z = pca.fit_transform(scaler.fit_transform(imputer.fit_transform(Xtr_)))
    # 先試 KMeans 分 2 群
    c = KMeans(n_clusters=2, n_init=10, random_state=random_state).fit_predict(Z)

    strat_key = ytr_.astype(str) + "|" + pd.Series(c, dtype=str)
    # 確保每個 (y,cluster) cell 的樣本數 >= n_splits；否則退而用 PC1 分箱
    ok = (strat_key.value_counts() >= n_splits).all()
    if not ok:
        pc1_bins = pd.qcut(Z[:,0], q=3, duplicates='drop', labels=False)
        strat_key = ytr_.astype(str) + "|" + pd.Series(pc1_bins, dtype=str)
        ok = (strat_key.value_counts() >= n_splits).all()
    # 還是不夠，就退回只分層 y（避免小 cell 讓分折失敗）
    if not ok:
        strat_key = ytr_.astype(str)

    return strat_key

strat_key_tr = build_strat_key_for_Xy(X_train, y_train, n_splits=5, random_state=42)

# --- 建立「同時顧到 y 與 X 結構」的 5 折 ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = [(tr_idx, va_idx) for tr_idx, va_idx in skf.split(np.zeros(len(y_train)), strat_key_tr)]

# --- 快速檢查每折分布 ---
c_summary = []
# 這裡的 cluster 僅用於檢查；若 strat_key 用 PC1 分箱或只 y，顯示會相應調整
for k, (_, va) in enumerate(folds, 1):
    y_va = y_train.iloc[va]
    # 從 strat_key 解析出 cluster（若沒有'|'就顯示N/A）
    key_va = strat_key_tr.iloc[va].astype(str)
    if key_va.str.contains(r'\|').any():
        cl_va = key_va.str.split('|').str[1].astype(int)
        c_summary.append(f"Fold{k}: y=1比例={y_va.mean():.3f} | cluster1比例={(cl_va==1).mean():.3f}")
    else:
        c_summary.append(f"Fold{k}: y=1比例={y_va.mean():.3f} | cluster=OnlyY")
print("\n".join(c_summary))


Fold1: y=1比例=0.068 | cluster1比例=0.331
Fold2: y=1比例=0.068 | cluster1比例=0.331
Fold3: y=1比例=0.068 | cluster1比例=0.335
Fold4: y=1比例=0.064 | cluster1比例=0.332
Fold5: y=1比例=0.064 | cluster1比例=0.336


In [24]:
# 用 X_train 估 Z、分群
pipe_X = KNNImputer(n_neighbors=10, weights='distance')
Z = PCA(n_components=2, random_state=42).fit_transform(
        StandardScaler().fit_transform(pipe_X.fit_transform(X_train)))
c = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(Z)

# 假設你已經有 c 與 y_train
c_s = pd.Series(c, index=y_train.index, name='cluster')  # 對齊索引！

p1_by_cluster = y_train.groupby(c_s).mean()
cluster_map = {p1_by_cluster.idxmax(): 1, p1_by_cluster.idxmin(): 0}
c_bin = c_s.map(cluster_map)

p1_high = y_train[c_bin == 1].mean()
p1_low  = y_train[c_bin == 0].mean()

eps = 1e-9
odds_ratio = ((p1_high+eps)/(1-p1_high+eps)) / ((p1_low+eps)/(1-p1_low+eps))
print("p(y=1|cluster):", p1_by_cluster.to_dict())
print("odds ratio(高p1群 vs 低p1群) =", odds_ratio)


p(y=1|cluster): {0: 0.0656525220176141, 1: 0.25}
odds ratio(高p1群 vs 低p1群) = 4.7439023844943


p(y=1|cluster): {0: 0.0656525220176141, 1: 0.25}
odds ratio(高p1群 vs 低p1群) = 4.7439023844943